# Training Notebook for weather_intelligence/train_weather.py

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import psycopg2
import json
import joblib
import os
import traceback
from datetime import datetime, timedelta

# Connection details
DB_DB = "postgres"
DB_USER = "postgres"
DB_PASS = "Madhavan@2004@"
DB_HOST = "db.lpoaswfbufnosziwhtsq.supabase.co"
DB_PORT = "5432"

def calculate_comfort_index(temp, humidity):
    return temp - 0.55 * (1 - 0.01 * humidity) * (temp - 14.5)

def main():
    print("--- STARTING WEATHER INTELLIGENCE K-FOLD TRAINING (XGBoost) ---")
    
    try:
        # 1. Load Data
        conn = psycopg2.connect(host=DB_HOST, database=DB_DB, user=DB_USER, password=DB_PASS, port=DB_PORT)
        weather_df = pd.read_sql("SELECT city_id, forecast_date, temp_min, temp_max, humidity, precipitation, alert_level FROM public.weather_forecasts", conn)
        sales_df = pd.read_sql("SELECT transaction_date, store_id, product_id, units_sold FROM public.sales_transactions", conn)
        stores_df = pd.read_sql("SELECT store_id, city_id FROM public.stores", conn)
        products_df = pd.read_sql("SELECT product_id, category_id FROM public.products", conn)
        conn.close()

        # 2. Pre-processing
        weather_df['forecast_date'] = pd.to_datetime(weather_df['forecast_date']).dt.date
        sales_df['transaction_date'] = pd.to_datetime(sales_df['transaction_date']).dt.date
        
        sales_df = sales_df.merge(stores_df, on='store_id')
        sales_df = sales_df.merge(products_df, on='product_id')
        
        # 3. FEATURE ENGINEERING
        climatology_norms = weather_df.groupby('city_id').agg({'temp_max': 'mean', 'precipitation': 'mean'}).reset_index()
        climatology_norms.rename(columns={'temp_max': 'seasonal_temp_norm', 'precipitation': 'seasonal_precip_norm'}, inplace=True)
        
        weather_df = weather_df.merge(climatology_norms, on='city_id')
        weather_df['temp_deviation'] = weather_df['temp_max'] - weather_df['seasonal_temp_norm']
        weather_df['precip_deviation'] = weather_df['precipitation'] - weather_df['seasonal_precip_norm']
        
        weather_df['comfort_index'] = weather_df.apply(lambda r: calculate_comfort_index(r['temp_max'], r['humidity']), axis=1)
        weather_df['heatwave_proxy'] = (weather_df['temp_max'] > 40).astype(int)
        weather_df['storm_proxy'] = (weather_df['precipitation'] > 50).astype(int)
        
        weather_df = weather_df.sort_values(['city_id', 'forecast_date'])
        for i in [1, 3]:
            weather_df[f'temp_dev_lag_{i}'] = weather_df.groupby('city_id')['temp_deviation'].shift(i)
            weather_df[f'precip_dev_lag_{i}'] = weather_df.groupby('city_id')['precip_deviation'].shift(i)

        # 4. TARGET REDEFINITION
        neutral_dates = weather_df[weather_df['alert_level'] == 'None'][['city_id', 'forecast_date']]
        baseline_data = sales_df.merge(neutral_dates, left_on=['city_id', 'transaction_date'], right_on=['city_id', 'forecast_date'])
        
        structural_baselines = baseline_data.groupby(['store_id', 'category_id'])['units_sold'].mean().reset_index()
        structural_baselines.rename(columns={'units_sold': 'baseline_units'}, inplace=True)
        
        dataset = sales_df.merge(weather_df, left_on=['city_id', 'transaction_date'], right_on=['city_id', 'forecast_date'])
        dataset = dataset.merge(structural_baselines, on=['store_id', 'category_id'])
        
        dataset['weather_attributed_delta'] = (dataset['units_sold'] - dataset['baseline_units']) / (dataset['baseline_units'] + 0.1)
        
        # 5. TRAINING PREP
        le_cat = LabelEncoder()
        dataset['category_enc'] = le_cat.fit_transform(dataset['category_id'].astype(str))
        joblib.dump(le_cat, 'ml/weather_intelligence/label_encoder_cat.pkl')
        
        features = [
            'category_enc', 'temp_max', 'temp_deviation', 'precip_deviation', 
            'comfort_index', 'heatwave_proxy', 'storm_proxy',
            'temp_dev_lag_1', 'temp_dev_lag_3', 'precip_dev_lag_1', 'precip_dev_lag_3'
        ]
        
        df = dataset.dropna(subset=features + ['weather_attributed_delta']).copy()
        
        X = df[features].values
        y = df['weather_attributed_delta'].values
        
        # Split: 80/20
        print(f"Dataset ready with {len(X):,} samples.")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
        
        # 6. K-Fold Cross Validation
        print("Performing 5-Fold Cross Validation...")
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_maes = []
        cv_r2s = []
        
        for train_idx, val_idx in kf.split(X_train):
            X_tr, y_tr = X_train[train_idx], y_train[train_idx]
            X_val, y_val = X_train[val_idx], y_train[val_idx]
            
            model_cv = xgb.XGBRegressor(
                n_estimators=300, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                objective='reg:squarederror', tree_method='hist', random_state=42
            )
            model_cv.fit(X_tr, y_tr)
            preds = model_cv.predict(X_val)
            
            cv_maes.append(mean_absolute_error(y_val, preds))
            cv_r2s.append(r2_score(y_val, preds))
            
        print(f"CV MAE: {np.mean(cv_maes):.4f}")
        print(f"CV R2: {np.mean(cv_r2s):.4f}")

        # 7. Final XGBOOST TRAINING
        print("Training final model on full train set...")
        final_model = xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8,
            objective='reg:squarederror', tree_method='hist', random_state=42
        )
        final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        
        # 8. EVALUATION
        y_pred = final_model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        report = {
            "model_name": "Weather Intelligence K-Fold XGBoost",
            "training_samples": len(X_train),
            "test_samples": len(X_test),
            "cross_validation_5fold": {
                "mean_MAE": f"{np.mean(cv_maes):.4f}",
                "mean_R2": f"{np.mean(cv_r2s):.4f}"
            },
            "holdout_test_metrics": {
                "MAE": f"{mae:.4f}",
                "R2_Score": f"{r2:.4f}"
            },
            "feature_importances": dict(zip(features, [float(i) for i in final_model.feature_importances_]))
        }
        
        print("\n--- OPTIMIZED WEATHER INTELLIGENCE REPORT ---")
        print(json.dumps(report, indent=4))
        
        if not os.path.exists('ml/weather_intelligence'):
            os.makedirs('ml/weather_intelligence')
            
        final_model.save_model('ml/weather_intelligence/weather_model_v2.json')
        with open('ml/weather_intelligence/training_report.json', 'w') as f:
            json.dump(report, f, indent=4)
            
        print("\n[SUCCESS] Optimized Weather Model trained and deployed.")

    except Exception:
        print("\n--- ERROR DURING TRAINING ---")
        traceback.print_exc()

if __name__ == "__main__":
    main()


--- STARTING WEATHER INTELLIGENCE K-FOLD TRAINING (XGBoost) ---


C:\Users\madha\AppData\Local\Temp\ipykernel_25228\2539334868.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  weather_df = pd.read_sql("SELECT city_id, forecast_date, temp_min, temp_max, humidity, precipitation, alert_level FROM public.weather_forecasts", conn)
C:\Users\madha\AppData\Local\Temp\ipykernel_25228\2539334868.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_df = pd.read_sql("SELECT transaction_date, store_id, product_id, units_sold FROM public.sales_transactions", conn)


C:\Users\madha\AppData\Local\Temp\ipykernel_25228\2539334868.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stores_df = pd.read_sql("SELECT store_id, city_id FROM public.stores", conn)
C:\Users\madha\AppData\Local\Temp\ipykernel_25228\2539334868.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products_df = pd.read_sql("SELECT product_id, category_id FROM public.products", conn)



--- ERROR DURING TRAINING ---


Traceback (most recent call last):
  File "C:\Users\madha\AppData\Local\Temp\ipykernel_25228\2539334868.py", line 75, in main
    joblib.dump(le_cat, 'ml/weather_intelligence/label_encoder_cat.pkl')
  File "C:\python\Lib\site-packages\joblib\numpy_pickle.py", line 552, in dump
    with open(filename, 'wb') as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'ml/weather_intelligence/label_encoder_cat.pkl'
